# Pair-score smoke test on Colab — expert = `kovi`

Trains **only** `PairScoreHead` (a single 2-layer MLP) on top of the **frozen** transformer_v1 encoder stack (`FleetEncoder` + `PlanetEncoder` + `PlanetEntityEncoder` + `CrossEntityAttention`). Single joint cross-entropy loss on flattened `(P×P)` pair logits, acted rows only, **one expert: `kovi`** — highest win-rate among the 5 sampled players in `data/replays/` (48.7% across 119 replays):

| player | replays | wins | win-rate |
|---|---:|---:|---:|
| **kovi** | **119** | **58** | **48.7%** |
| Shun_PI | 117 | 51 | 43.6% |
| bowwowforeach | 99 | 39 | 39.4% |
| Erfan Eshratifar | 73 | 26 | 35.6% |
| Orbital Occle | 238 | 78 | 32.8% |

**Goal.** Answer one question:

> Can the current frozen encoder representation support direct expert `(source, target)` pair prediction from kovi's replays?

If yes → we layer NOOP / frac / value / PPO back on top. If the head can't even overfit a tiny subset → fix the encoder/labels/masking before doing anything else.

**Prerequisites in `gs://orbit-wars-shipping/`:**
1. `code.tgz`, `data.tgz`, `weights.tgz` — built by `scripts/pack_for_gpu.sh`.
2. **`pair_score_assets.tgz`** — extra bundle this notebook needs (≈15 MB). The standard `weights.tgz` carries only the fleet/planet/entity encoder ckpts and `data.tgz` doesn't include `data/replays/`. Build & upload from your local repo:
   ```bash
   tar czf /tmp/pair_score_assets.tgz \
       data/runs/action/<run>/action_best.pt data/replays/kovi
   gsutil cp /tmp/pair_score_assets.tgz gs://orbit-wars-shipping/pair_score_assets.tgz
   ```
   (Pick the action run you want as the encoder source — e.g. `20260505-143435`.)

**Runtime:** Runtime → Change runtime type → **T4 GPU** (or higher).

Total runtime: ~3 min setup + ~2 min Experiment 1 (tiny-overfit) + ~10 min Experiment 2 (small-real) ≈ 15 min.

## 1. Verify GPU

In [ ]:
import torch, sys
if not torch.cuda.is_available():
    sys.exit('No GPU runtime — Runtime → Change runtime type → T4 GPU, then re-run.')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'CUDA: {torch.version.cuda}  PyTorch: {torch.__version__}')

## 2. Authenticate to GCP & pull tarballs

In [ ]:
from google.colab import auth
auth.authenticate_user()

PROJECT = 'analog-receiver-489214-e9'
BUCKET  = 'gs://orbit-wars-shipping'
PLAYER  = 'kovi'   # expert

!gcloud config set project {PROJECT}

In [ ]:
import os
WORK = '/content/orbit-wars'
os.makedirs(WORK, exist_ok=True)
%cd {WORK}

for name in ('code.tgz', 'data.tgz', 'weights.tgz'):
    !gsutil cp {BUCKET}/{name} .

## 3. Unpack + install

In [ ]:
%cd {WORK}
!tar xzf code.tgz
!tar xzf data.tgz
!tar xzf weights.tgz

# Pair-score asset bundle: action_best.pt (encoder ckpt) + data/replays/kovi/.
# The standard `weights.tgz` only carries fleet/planet/entity encoder ckpts;
# pair_score needs a saved action stack (it loads only the encoder keys),
# and `--player kovi` needs the replay tree to map replay-id → CSV. This
# extra tarball plugs both gaps. Build & upload from your local repo with:
#
#   tar czf /tmp/pair_score_assets.tgz \
#       data/runs/action/<run>/action_best.pt data/replays/kovi
#   gsutil cp /tmp/pair_score_assets.tgz {BUCKET}/pair_score_assets.tgz
!gsutil cp {BUCKET}/pair_score_assets.tgz . && tar xzf pair_score_assets.tgz

import glob
act = sorted(glob.glob('data/runs/action/*/action_best.pt'))
if not act:
    act = sorted(glob.glob('data/runs/action/*/action_last.pt'))
if not act:
    raise SystemExit(
        'no action_*.pt found — upload pair_score_assets.tgz to '
        f'{BUCKET} (see the comment above for the build command).'
    )
ENCODER_CKPT = act[-1]
print('encoder ckpt:', ENCODER_CKPT)

# Confirm kovi replays are present so --player kovi can resolve.
kovi = sorted(glob.glob(f'data/replays/{PLAYER}/*.json.gz'))
if not kovi:
    raise SystemExit(
        f'no replays under data/replays/{PLAYER}/ — '
        'pair_score_assets.tgz must include data/replays/<player>/.'
    )
print(f'replays for {PLAYER}: {len(kovi)}')

In [ ]:
%cd {WORK}
!pip install -q -r requirements.txt --no-deps
!pip install -q kaggle-environments

## 4. Sanity-check imports + dataset coverage

In [ ]:
import sys
sys.path.insert(0, WORK)

from agents.transformer_v1.pretrain.pair_score import (
    PairScoreHead, PairScoreStack, compute_pair_score_loss,
    discover_action_csvs, load_frozen_encoder_stack, player_replay_stems,
)
from agents.transformer_v1.pretrain.expert_action import ActionSnapshotDataset
from agents.transformer_v1.paths import (
    ACTION_DATASET_DIR, PLANET_DATASET_DIR, FLEET_DATASET_DIR,
    ENTITY_DATASET_DIR, CROSS_ENTITY_DATASET_DIR,
)
from pathlib import Path

REPLAY_DIR = Path('data/replays')
kovi_stems = player_replay_stems(REPLAY_DIR, PLAYER)
kovi_csvs  = discover_action_csvs(
    Path(ACTION_DATASET_DIR), filter_mode='all',
    player=PLAYER, replay_dir=REPLAY_DIR,
)
kovi_winner_csvs = discover_action_csvs(
    Path(ACTION_DATASET_DIR), filter_mode='winner',
    player=PLAYER, replay_dir=REPLAY_DIR,
)
print(f'replays for {PLAYER}:                 {len(kovi_stems)}')
print(f'action CSVs for {PLAYER} (all):       {len(kovi_csvs)}')
print(f'action CSVs for {PLAYER} (winner):    {len(kovi_winner_csvs)}')

# Coverage check — stems present across all 5 CSV dirs.
stems = {p.stem.removeprefix('action_') for p in kovi_csvs}
for d, prefix in (
    (PLANET_DATASET_DIR, 'planet_'), (FLEET_DATASET_DIR, 'fleet_'),
    (ENTITY_DATASET_DIR, 'entity_'), (CROSS_ENTITY_DATASET_DIR, 'cross_entity_'),
):
    stems &= {p.stem.removeprefix(prefix) for p in Path(d).glob(f'{prefix}*.csv')}
print(f'{PLAYER} CSVs covered by all 5 dirs: {len(stems)}')

## 5. Experiment 1 — tiny-overfit (BLOCKING gate)

Train on 50 acted rows from kovi, train==val. Should reach `train_loss < 0.1` and `top1 > 0.9` within ~150 epochs. **If this fails, STOP** — likely a mask / label-index / flatten / encoder-representation bug.

In [ ]:
import time
TS = time.strftime('%Y%m%d-%H%M%S')
OUT_OVERFIT = f'data/runs/pair_score/overfit_{PLAYER}_{TS}'

%cd {WORK}
!python -m agents.transformer_v1.pretrain.pair_score \
    --encoder-ckpt {ENCODER_CKPT} \
    --player {PLAYER} --filter all \
    --max-rows 50 --overfit \
    --batch-size 32 --lr 1e-3 --epochs 150 \
    --device cuda \
    --out-dir {OUT_OVERFIT}

In [ ]:
import json
log = json.loads(open(f'{WORK}/{OUT_OVERFIT}/log.json').read())
last = log[-1]
print(f"epoch {last['epoch']}  tr_loss={last['train']['loss']:.4f}  tr_top1={last['train']['top1']:.3f}  val_top1={last['val']['top1']:.3f}")
if last['train']['loss'] > 0.1 or last['train']['top1'] < 0.9:
    print('\n⚠ overfit gate not cleared — investigate before Experiment 2.')
else:
    print('\n✓ overfit gate cleared — proceed to Experiment 2.')

## 6. Experiment 2 — small-real split (kovi only)

Up to 5000 acted rows from kovi's perspective, 80/20 train/val. Track top-1 / top-3 / top-5 vs the random-valid baseline. **Minimal success:** val top-1 ≥ 3× random. **Strong success:** val top-1 ≥ 0.30, top-3 ≥ 0.55.

In [ ]:
TS = time.strftime('%Y%m%d-%H%M%S')
OUT_SMALL = f'data/runs/pair_score/small_{PLAYER}_{TS}'

%cd {WORK}
!python -m agents.transformer_v1.pretrain.pair_score \
    --encoder-ckpt {ENCODER_CKPT} \
    --player {PLAYER} --filter all \
    --max-rows 5000 --val-frac 0.2 \
    --batch-size 64 --lr 1e-3 --epochs 10 \
    --device cuda \
    --out-dir {OUT_SMALL}

In [ ]:
import json
log = json.loads(open(f'{WORK}/{OUT_SMALL}/log.json').read())
for e in log:
    v = e['val']
    rand = v.get('random_valid_top1', 0.0)
    margin = v['top1'] / max(rand, 1e-6)
    print(
        f"ep {e['epoch']:2d}  tr_loss={e['train']['loss']:.3f}  "
        f"val_loss={v['loss']:.3f}  val_top1={v['top1']:.3f}  "
        f"top3={v['top3']:.3f}  top5={v['top5']:.3f}  "
        f"rand={rand:.3f}  margin={margin:.1f}x"
    )
best = max(log, key=lambda e: e['val']['top1'])
print(
    f"\nbest val_top1={best['val']['top1']:.3f} "
    f"(epoch {best['epoch']}); rand={best['val'].get('random_valid_top1', 0.0):.3f}"
)

## 7. Push results to GCS

In [ ]:
%cd {WORK}
!gsutil -m cp -r {OUT_OVERFIT} {BUCKET}/runs/
!gsutil -m cp -r {OUT_SMALL}   {BUCKET}/runs/
print('uploaded:')
print(f'  {BUCKET}/runs/{OUT_OVERFIT.rsplit("/", 1)[-1]}')
print(f'  {BUCKET}/runs/{OUT_SMALL.rsplit("/", 1)[-1]}')

## 8. Decision tree

| Outcome | Read | Next step |
|---|---|---|
| Experiment 1 fails (no overfit) | label / mask / flatten / encoder bug | debug **before** any further training |
| Experiment 2 ≤ 1× random | encoder representation insufficient | revisit L1/L2 (PlanetEntityEncoder, CrossEncoder) — add capacity, fix mask coverage, or change pair feature |
| Experiment 2 ≥ 3× random | representation OK | layer NOOP head + frac head + value head back on top, then re-attempt PPO |
| Experiment 2 between | weak signal | try `--filter winner` on kovi (kovi-wins-only), or pick a different player |
